In [1]:
# 03_model_training_and_selection.ipynb
# Sex-stratified risk classification. Compares seven algorithms (XGBoost, CatBoost,
# LightGBM, RandomForest, ExtraTrees, SVM, MLP) with Optuna-tuned tree ensembles,
# evaluates on a held-out test set, runs DeLong tests of XGBoost vs each competitor,
# and persists the selected XGBoost models, data splits, and test predictions for
# downstream notebooks. Explicit split scheme: per-sex stratified 80/20 train/test;
# hyperparameters tuned by 3-fold CV (AUC) on the training fold only.

import warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             accuracy_score, f1_score)
import scipy.stats

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"; TAB = ROOT / "results" / "tables"
MODELS = ROOT / "models"; MODELS.mkdir(exist_ok=True)

df = pd.read_parquet(DATA / "analytic_dataset.parquet")
PRED = ["age", "HE_BMI", "HE_wc", "HE_wt", "HE_ht", "N_EN", "N_CHO", "N_SUGAR",
        "N_CHOL", "sm_presnt", "dr_month", "pa_aerobic", "L_BR_FQ", "edu", "ho_incm", "D_1_1"]

# ---- DeLong test for two correlated ROC AUCs ----
def delong_test(y, p1, p2):
    y = np.asarray(y); order = np.argsort(-y)  # positives first not required; use fast algorithm
    def compute_midrank(x):
        J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N)
        i = 0
        while i < N:
            j = i
            while j < N and Z[j] == Z[i]:
                j += 1
            T[i:j] = 0.5 * (i + j - 1) + 1
            i = j
        T2 = np.empty(N); T2[J] = T
        return T2
    y = np.asarray(y)
    pos = y == 1; neg = y == 0
    m = pos.sum(); n = neg.sum()
    preds = np.vstack([p1, p2])
    tx = preds[:, pos]; ty = preds[:, neg]
    k = 2
    tz = np.concatenate([tx, ty], axis=1)
    v01 = np.empty((k, m)); v10 = np.empty((k, n))
    aucs = np.empty(k)
    for r in range(k):
        mrx = compute_midrank(tx[r]); mry = compute_midrank(ty[r])
        mrz = compute_midrank(tz[r])
        aucs[r] = (mrz[:m].sum() - m * (m + 1) / 2) / (m * n)
        v01[r] = (mrz[:m] - mrx) / n
        v10[r] = 1.0 - (mrz[m:] - mry) / m
    sx = np.cov(v01); sy = np.cov(v10)
    s = sx / m + sy / n
    d = aucs[0] - aucs[1]
    var = s[0, 0] + s[1, 1] - 2 * s[0, 1]
    if var <= 0:
        return aucs[0], aucs[1], d, 1.0
    z = d / np.sqrt(var)
    p = 2 * (1 - scipy.stats.norm.cdf(abs(z)))
    return aucs[0], aucs[1], d, p

def optuna_tree(kind, Xtr, ytr, n_trials=25):
    def objective(trial):
        if kind == "xgb":
            params = dict(max_depth=trial.suggest_int("max_depth", 2, 7),
                          learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                          n_estimators=trial.suggest_int("n_estimators", 100, 600),
                          subsample=trial.suggest_float("subsample", 0.6, 1.0),
                          colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
                          min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
                          gamma=trial.suggest_float("gamma", 0, 5))
            clf = xgb.XGBClassifier(**params, eval_metric="logloss", random_state=SEED, n_jobs=4)
        elif kind == "cat":
            params = dict(depth=trial.suggest_int("depth", 2, 8),
                          learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                          iterations=trial.suggest_int("iterations", 100, 600),
                          l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 10))
            clf = CatBoostClassifier(**params, random_state=SEED, verbose=0)
        elif kind == "lgbm":
            params = dict(max_depth=trial.suggest_int("max_depth", 2, 8),
                          learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                          n_estimators=trial.suggest_int("n_estimators", 100, 600),
                          num_leaves=trial.suggest_int("num_leaves", 15, 127),
                          subsample=trial.suggest_float("subsample", 0.6, 1.0))
            clf = LGBMClassifier(**params, random_state=SEED, n_jobs=4, verbose=-1)
        elif kind == "rf":
            params = dict(n_estimators=trial.suggest_int("n_estimators", 100, 500),
                          max_depth=trial.suggest_int("max_depth", 3, 20),
                          min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20))
            clf = RandomForestClassifier(**params, random_state=SEED, n_jobs=4)
        elif kind == "et":
            params = dict(n_estimators=trial.suggest_int("n_estimators", 100, 500),
                          max_depth=trial.suggest_int("max_depth", 3, 20),
                          min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20))
            clf = ExtraTreesClassifier(**params, random_state=SEED, n_jobs=4)
        cv = StratifiedKFold(3, shuffle=True, random_state=SEED)
        return cross_val_score(clf, Xtr, ytr, cv=cv, scoring="roc_auc", n_jobs=1).mean()
    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

def build(kind, params):
    if kind == "xgb":
        return xgb.XGBClassifier(**params, eval_metric="logloss", random_state=SEED, n_jobs=4)
    if kind == "cat":
        return CatBoostClassifier(**params, random_state=SEED, verbose=0)
    if kind == "lgbm":
        return LGBMClassifier(**params, random_state=SEED, n_jobs=4, verbose=-1)
    if kind == "rf":
        return RandomForestClassifier(**params, random_state=SEED, n_jobs=4)
    if kind == "et":
        return ExtraTreesClassifier(**params, random_state=SEED, n_jobs=4)
    if kind == "svm":
        return make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEED))
    if kind == "mlp":
        return make_pipeline(StandardScaler(),
                             MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=400,
                                           early_stopping=True, random_state=SEED))

NAMES = {"xgb": "XGBoost", "cat": "CatBoost", "lgbm": "LightGBM", "rf": "RandomForest",
         "et": "ExtraTrees", "svm": "SVM", "mlp": "NeuralNet"}
TUNE = ["xgb", "cat", "lgbm", "rf", "et"]       # Optuna-tuned tree ensembles
FIXED = ["svm", "mlp"]                           # standard configurations

results = []; test_store = {}; split_store = {}
for sx in ["Male", "Female"]:
    s = df[df.sex == sx].reset_index(drop=True)
    X = s[PRED].values; y = s["diabetes"].values
    Xtr, Xte, ytr, yte, idx_tr, idx_te = train_test_split(
        X, y, s.index.values, test_size=0.2, stratify=y, random_state=SEED)
    split_flag = np.where(np.isin(s.index.values, idx_tr), "train", "test")
    split_store[sx] = s.assign(split=split_flag)
    preds_sex = {"y_true": yte}
    best_params_sex = {}
    for kind in TUNE + FIXED:
        params = optuna_tree(kind, Xtr, ytr, n_trials=25) if kind in TUNE else {}
        best_params_sex[kind] = params
        clf = build(kind, params).fit(Xtr, ytr)
        p = clf.predict_proba(Xte)[:, 1]
        preds_sex[kind] = p
        # optimal threshold = argmax F1 over predicted-prob grid
        grid = np.linspace(0.05, 0.95, 91)
        f1s = [f1_score(yte, (p >= t).astype(int)) for t in grid]
        thr = grid[int(np.argmax(f1s))]
        results.append([sx, NAMES[kind], accuracy_score(yte, (p >= thr).astype(int)),
                        f1_score(yte, (p >= thr).astype(int)), roc_auc_score(yte, p),
                        average_precision_score(yte, p), brier_score_loss(yte, p), round(thr, 2)])
        if kind == "xgb":
            clf.save_model(MODELS / f"xgb_{sx.lower()}.json")
            with open(MODELS / f"xgb_{sx.lower()}_params.json", "w") as f:
                json.dump(params, f, indent=2)
    test_store[sx] = pd.DataFrame(preds_sex)
    print(f"[{sx}] trained 7 models on {len(ytr)} train / {len(yte)} test")

perf = pd.DataFrame(results, columns=["Sex", "Model", "Accuracy", "F1", "AUC",
                                      "PR_AUC", "Brier", "OptThreshold"]).round(3)
perf.to_csv(TAB / "table_algorithm_comparison.csv", index=False)
print("\nSeven-algorithm comparison (held-out test):")
print(perf.to_string(index=False))

# ---- DeLong: XGBoost vs each competitor (per sex) ----
dl = []
for sx in ["Male", "Female"]:
    ts = test_store[sx]; y = ts["y_true"].values
    for kind in ["cat", "lgbm", "rf", "et", "svm", "mlp"]:
        a1, a2, d, p = delong_test(y, ts["xgb"].values, ts[kind].values)
        dl.append([sx, "XGBoost vs " + NAMES[kind], round(a1, 3), round(a2, 3),
                   round(d, 3), f"{p:.3f}"])
dlf = pd.DataFrame(dl, columns=["Sex", "Comparison", "AUC_XGB", "AUC_other", "Delta", "DeLong_p"])
dlf.to_csv(TAB / "table_delong_tests.csv", index=False)
print("\nDeLong tests (XGBoost vs competitors):")
print(dlf.to_string(index=False))

# ---- Persist XGBoost test predictions and splits for downstream notebooks ----
for sx in ["Male", "Female"]:
    split_store[sx].to_parquet(DATA / f"split_{sx.lower()}.parquet", index=False)
    test_store[sx][["y_true", "xgb"]].rename(columns={"xgb": "xgb_prob"}).to_parquet(
        DATA / f"test_pred_{sx.lower()}.parquet", index=False)

xgb_rows = perf[perf.Model == "XGBoost"]
print("\nSelected model = XGBoost (DiCE-compatible; competitive AUC). Summary:")
print(xgb_rows.to_string(index=False))
print("\nSaved models/, algorithm comparison, DeLong tests, splits, test predictions.")


[Male] trained 7 models on 7999 train / 2000 test


[Female] trained 7 models on 10484 train / 2621 test

Seven-algorithm comparison (held-out test):
   Sex        Model  Accuracy    F1   AUC  PR_AUC  Brier  OptThreshold
  Male      XGBoost     0.676 0.467 0.772   0.408  0.132          0.22
  Male     CatBoost     0.694 0.470 0.772   0.414  0.132          0.23
  Male     LightGBM     0.669 0.475 0.768   0.402  0.133          0.22
  Male RandomForest     0.704 0.477 0.767   0.402  0.133          0.25
  Male   ExtraTrees     0.722 0.461 0.766   0.399  0.134          0.26
  Male          SVM     0.698 0.417 0.707   0.352  0.145          0.19
  Male    NeuralNet     0.688 0.460 0.768   0.401  0.133          0.24
Female      XGBoost     0.769 0.451 0.821   0.385  0.098          0.21
Female     CatBoost     0.769 0.446 0.821   0.379  0.098          0.21
Female     LightGBM     0.767 0.451 0.819   0.374  0.098          0.21
Female RandomForest     0.773 0.453 0.821   0.370  0.098          0.22
Female   ExtraTrees     0.794 0.440 0.814   0.368 